# Data Collection: College Sailing Scores Scraper

This notebook scrapes regatta, sailor, and scoring data from College Sailing Scores
and appends results into `races.csv` for downstream analysis and modeling.

In [5]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import numpy as np

## 1. Season Index + Regatta List
Fetch regatta links for each season and build a list of regattas to scrape.

In [8]:
# Load existing dataset if present; otherwise initialize an empty schema
seasons = ['f24', 's24', 'f23', 's23', 'f22','s22']

df_races = pd.DataFrame()

try:
    df_races = pd.read_csv("races.csv")
except FileNotFoundError:
    df_races = pd.DataFrame(
        columns=[
            "Season", "Regatta", "Race", "Division",
            "Position", "Sailor", "Partner", "Team",
            "Venue", "Score"
        ]
    )

regattas = {}
for season in seasons:
  url = f"https://scores.collegesailing.org/{season}/"
  page = requests.get(url)
  listSoup = BeautifulSoup(page.content, 'html.parser')
  
  tbody = listSoup.find('table', class_="season-summary").find('tbody')
  
  for link in tbody.find_all("a", href=True):
    if (season + "/" + link['href']) not in df_races['Regatta'].unique():
      scoring = link.parent.next_sibling.next_sibling.next_sibling.text
      if (scoring == "3 Divisions" or scoring == "2 Divisions" or scoring == "Combined"):
        regattas[season + "/" + link['href']] = {"link":season + "/" + link['href'], "scoring":scoring}

## 2. Fetch Regatta Pages
For each regatta, download and store the HTML needed for later parsing.

In [11]:
regattaSoups = {}

for i, regatta in enumerate(list(regattas.values())):
  betterVenue = list(regattas.keys())[i]
  print(f"[{i + 1}/{len(regattas)}] Fetching HTML for {betterVenue}")
    
  # full scores
  url = f"https://scores.collegesailing.org/{regatta['link']}/full-scores/"
  page = requests.get(url)
  fullScores = BeautifulSoup(page.content, 'html.parser')

  # sailors
  url = f"https://scores.collegesailing.org/{regatta['link']}/sailors/"
  page = requests.get(url)
  sailors = BeautifulSoup(page.content, 'html.parser')
  
  regattaSoups[betterVenue] = {"fullScores": fullScores, "sailors": sailors, "scoring": regatta['scoring']}

[1/1] Fetching HTML for f24/rose-bowl-2025


## 3. Helper Functions
Utilities to normalize race ranges and convert parsed results into row format.

In [42]:
def getRaceNums(oldNums, scoresLen):
    """Convert race number ranges (e.g., '1-3') into a flat list of race numbers."""
    newNums = []
    if oldNums == [['']]:
        newNums = list(range(1, scoresLen + 1))
    elif len(oldNums) > 0:
        for i, num in enumerate(oldNums):
            if len(num) > 1:
                for j in range(int(num[0]), int(num[1]) + 1):
                    newNums.append(j)
            else:
                newNums.append(int(num[0]))
    return newNums
def makeRaceSeries(score, team, raceNum, division, name, position, partner, venue, regatta, teams, date):
    """Create a standardized race result record for appending to the master dataset."""
    raceSeries = pd.Series()
    raceSeries['raceID'] = "" + regatta + "/" + str(raceNum) + division
    if isinstance(score, int):
        raceSeries["Score"] = score
    else:
        raceSeries["Score"] = len(teams) + 1
    raceSeries["Date"] = date
    raceSeries["Div"] = division
    raceSeries["Sailor"] = name
    raceSeries["Position"] = position
    raceSeries["Partner"] = partner
    raceSeries["Team"] = team
    raceSeries["Venue"] = venue
    raceSeries["Regatta"] = regatta
    raceSeries["Teams"] = teams
    return raceSeries

## 4. Parse + Append Results
Parse the stored HTML for each regatta and append race-level rows to `df_races`.

In [43]:
# Parse each regatta page and append race-level results to the dataset.
for i, regatta in enumerate(list(regattaSoups.keys())):
    print(f"[{i + 1}/{len(regattaSoups)}] Parsing {regatta}")
    fullScores = regattaSoups[regatta]['fullScores']
    sailors = regattaSoups[regatta]['sailors']
    scoring = regattaSoups[regatta]['scoring']
    
    if len(fullScores.find_all('table', class_="results")) == 0: 
        print(f"Skipping {regatta}: no scores posted yet")
        continue
    
    scoreData = fullScores.find_all('table', class_="results")[
        0].contents[1].contents
    header = fullScores.find(
        'table', class_="results").find_all('th', class_="right")
    raceCount = int(header[len(header) - 2].text)
        
    
    numDivisions = 1
    if scoreData[1]['class'][0] == 'divB' and scoreData[2]['class'][0] == 'totalrow':
        numDivisions = 2
    if scoreData[2]['class'][0] == 'divC':
        numDivisions = 3


    teamCount = int(len(scoreData) / (numDivisions + 1))
    
    teamHomes = [(scoreData[(k*(numDivisions + 1)) - (numDivisions + 1)].find('a').text)
                 for k in range(teamCount)]
    
    host = fullScores.find("span", itemprop='location').text
    date = fullScores.find("time").attrs['datetime']
    date = date[:10]
    
    if scoring == "Combined":
        teamHomes = teamHomes * numDivisions

    for i in range(1, teamCount):
        teamHome = scoreData[(i*(numDivisions + 1)) - (numDivisions + 1)].find('a').text
        teamName = scoreData[(i*(numDivisions + 1)) - (numDivisions + 1) + 1].contents[2].text
        teamScores = {'A': [], 'B': [], 'C':[]}

        teamScores["A"] = [int(scoreData[(i*(numDivisions + 1)) - (numDivisions + 1)].contents[j].text) for j in range(
            4, (4 + raceCount)) if scoreData[(i*(numDivisions + 1)) - (numDivisions + 1)].contents[j].text.isdigit()]
        if numDivisions > 1:
            teamScores["B"] = [int(scoreData[(i*(numDivisions + 1)) - (numDivisions + 1) + 1].contents[j].text) for j in range(
                4, (4 + raceCount)) if scoreData[(i*(numDivisions + 1)) - (numDivisions + 1) + 1].contents[j].text.isdigit()]
        if numDivisions > 2:
            teamScores["C"] = [int(scoreData[(i*(numDivisions + 1)) - (numDivisions + 1) + 2].contents[j].text) for j in range(
                4, (4 + raceCount)) if scoreData[(i*(numDivisions + 1)) - (numDivisions + 1) + 2].contents[j].text.isdigit()]

        teamNameEls = [i for i in sailors.find_all(
            'td', class_="teamname") if i.text == teamName]
        
        if len(teamNameEls) == 0:
            print("team name entered wrong. Skipping team", teamName)
            continue
        
        teamNameEl = teamNameEls[0]

        rowClass = teamNameEl.parent['class'][1]

        index = 0
        row = teamNameEl.parent
        
        prevSkipper = ""
        prevCrew = ""
        
        while row.next_sibling is not None and row['class'][0] != "topborder" and row['class'][0] != "reserves-row" or index == 0:
            curRow = row
            while curRow.find_all('td', class_="division-cell") == []:
                curRow = curRow.previous_sibling
            division = curRow.find_all('td', class_="division-cell")[0].text

            # Get Skipper
            skipper = row.contents[len(row.contents) - 4]
            skipperName = skipper.text.split(" '", 1)[0]
            
            if skipperName == "No show":
                skipperName = ""

            # Get Crew
            crew = row.contents[len(row.contents) - 2]
            crewName = crew.text.split(" '", 1)[0]
            
            if crewName == "No show":
                crewName = ""
            
            if skipperName != "" and crewName != "":
                skipperRaceNums = skipper.next_sibling.text.split(",")
                skipperRaceNums = getRaceNums([i.split("-", 1) for i in skipperRaceNums], len(teamScores[division]))
                if len(skipperRaceNums) == 0:
                    print(f"skipper {skipperName} sailed no races? skipping")
                    row = row.next_sibling
                    index += 1
                    continue
                
                crewRaceNums = crew.next_sibling.text.split(",")
                crewRaceNums = getRaceNums([i.split("-", 1) for i in crewRaceNums], len(teamScores[division]))
                
                if len(crewRaceNums) == 0:
                    print(f"crew {crewName} sailed no races? skipping")
                    row = row.next_sibling
                    index += 1
                    continue
                
                
                skipperPartners = [(crewName if curRace in crewRaceNums else "Unknown") for curRace in range(1, skipperRaceNums[-1] + 1)]
                crewPartners = [(skipperName if curRace in skipperRaceNums else "Unknown") for curRace in range(1, crewRaceNums[-1] + 1)]
                
                for i, score in enumerate(teamScores[division]):
                    if i + 1 in skipperRaceNums:
                        df_races = pd.concat([df_races, makeRaceSeries(score, teamHome, i + 1, division, skipperName, "Skipper", skipperPartners[i], host,regatta,[t for t in teamHomes], date).to_frame().T])
                    if i + 1 in crewRaceNums:
                        df_races = pd.concat([df_races, makeRaceSeries(score, teamHome, i + 1, division, crewName, "Crew", crewPartners[i], host,regatta,[t for t in teamHomes],date).to_frame().T])
                
                prevSkipper = skipperName
                prevCrew = crewName
            
            elif skipperName != "":
                raceNums = skipper.next_sibling.text.split(",")
                raceNums = getRaceNums([i.split("-", 1) for i in raceNums], len(teamScores[division]))

                # Edge case: some pages list only skipper or only crew for a set of races.
                # In those cases, partner is recorded as "Unknown" to keep row format consistent.


                for i, score in enumerate(teamScores[division]):
                    if i + 1 in raceNums:
                        df_races = pd.concat([df_races, makeRaceSeries(score, teamHome, i + 1, division, skipperName, "Skipper", "Unknown", host,regatta,[t for t in teamHomes],date).to_frame().T])
                
                        
                prevSkipper = skipperName
                
            elif crewName != "":
                raceNums = crew.next_sibling.text.split(",")
                raceNums = getRaceNums([i.split("-", 1) for i in raceNums], len(teamScores[division]))
                
                # Still need prev person
                
                for i, score in enumerate(teamScores[division]):
                    if i + 1 in raceNums:
                        df_races = pd.concat([df_races, makeRaceSeries( score, teamHome, i + 1, division, crewName, "Crew", "Unknown", host,regatta,[t for t in teamHomes],date).to_frame().T])
    
                    
                prevCrew = crewName

            row = row.next_sibling
            index += 1

(1/327) analyzing s23/sailing-dinghy-national
(2/327) analyzing s23/sailing-open-west-semis
(3/327) analyzing s23/sailing-open-east-semis
(4/327) analyzing s23/ucsd-intrasquads
(5/327) analyzing s23/sailing-women-national
(6/327) analyzing s23/sailing-women-west-semis
(7/327) analyzing s23/sailing-women-east-semis
(8/327) analyzing s23/kiara-broudy
(9/327) analyzing s23/tufts-alumni-2023-420s
skipper Jacob Witney sailed no races? skipping
(10/327) analyzing s23/tufts-alumni-2023-larks
(11/327) analyzing s23/engineer-cup
(12/327) analyzing s23/gorge-invite
(13/327) analyzing s23/nu-spring
(14/327) analyzing s23/drexel-open
(15/327) analyzing s23/america
(16/327) analyzing s23/sailing-national-invitational
(17/327) analyzing s23/cream-city-classic
(18/327) analyzing s23/south-fleet-race
(19/327) analyzing s23/little
(20/327) analyzing s23/open-new-england-fleet-race
(21/327) analyzing s23/wesleyan-invite
(22/327) analyzing s23/pacific-coast-open-fleet-race-championships
(23/327) analyzin

## 5. Export
Write the updated dataset to disk for analysis notebooks.


In [ ]:
df_races.to_csv("races.csv", index=False)